06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [7]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch

# WORKING WITH 
datasetPath = 'data/clothDataset_5_.csv'

cloth_info = pd.read_csv(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (42, 326)
cloth_info: 
    frame         x0        y0         z0         vx0       vy0        vz0  \
0       0   0.140004 -48.67612  24.866420   -19.50266  2408.146 -1243.0920   
1       1   0.140004 -48.66430  24.862490   -19.50070  2408.157 -1242.7810   
2       2   0.140004 -48.66108  24.863770   -19.50134  2407.857 -1243.0400   
3       3   0.140004 -48.67768  24.860680   -19.49979  2408.397 -1242.8230   
4       4   0.140004 -48.66790  24.868310   -19.50361  2408.671 -1243.2230   
5       5   0.140004 -48.67289  24.860630   -19.49977  2408.343 -1243.0590   
6       6   0.140004 -48.66142  24.866240   -19.50257  2408.141 -1243.2120   
7       7   0.140004 -48.66218  24.859890   -19.49940  2408.175 -1243.1860   
8       8   0.140004 -48.66777  24.850430   -19.49467  2408.360 -1242.8460   
9       9   0.140004 -48.66874  24.851690   -19.49530  2408.776 -1242.8540   
10     10   0.140004 -48.67132  24.868500   -19.50371  2408.692 -1243.4100   
11     11   0.140004 -4

In [8]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=25):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        if isinstance(csv_data, str) and "frame,x0" in csv_data:
            self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        else:
            self.data = pd.read_csv(csv_data)
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']

        self.position_prefixes = ['x', 'y', 'z']
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')

        # Quitamos primera fila de outputs (no es el output de nada) y ultima fila de input (no tiene output)
        self.output_positions = self.output_positions.iloc[1:]
        self.data = self.data.iloc[:-1]
        

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)
    
    def num_features(self):
        # Number of columns in a row (pos, vel, sdf, uv per vertex)
        return len(self.feature_prefixes)
    
    def num_vertex(self):
        return self.num_vertices
    
    def _get_frame_output_tensor(self, idx):
         row = self.output_positions.iloc[idx]
         frame_data = []
         for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.position_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

         tensor_data = torch.tensor(np.array(frame_data))

         return tensor_data
    
    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_output_tensor(idx) # +1 ya no TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(datasetPath)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break

Batch Shape: torch.Size([4, 25, 13])
tensor([[[  0.1400, -48.6713,  24.8685,  ...,   1.0000,   0.7500,   0.0000],
         [  0.1400, -23.6700,  49.8703,  ...,   1.0000,   1.0000,   0.2500],
         [  0.1400, -48.6681,  49.8690,  ...,   1.0000,   1.0000,   0.0000],
         ...,
         [  0.1400,  26.3268, -50.1339,  ...,   1.0000,   0.0000,   0.7500],
         [  0.1400,  51.3300, -25.1385,  ...,   0.0000,   0.2500,   1.0000],
         [  0.1400,  51.3300, -50.1385,  ...,   0.0000,   0.0000,   1.0000]],

        [[  0.1400, -48.6764,  24.8574,  ...,   1.0000,   0.7500,   0.0000],
         [  0.1400, -23.6711,  49.8579,  ...,   1.0000,   1.0000,   0.2500],
         [  0.1400, -48.6798,  49.8559,  ...,   1.0000,   1.0000,   0.0000],
         ...,
         [  0.1400,  26.3297, -50.1428,  ...,   1.0000,   0.0000,   0.7500],
         [  0.1400,  51.3300, -25.1385,  ...,   0.0000,   0.2500,   1.0000],
         [  0.1400,  51.3300, -50.1385,  ...,   0.0000,   0.0000,   1.0000]],

       

In [12]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        #print(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas

# nn.Linear in PyTorch is designed to handle 3D tensors seamlessly.  
# When a 3D input tensor (e.g., batch_size, sequence_length, features) is provided,
#  the layer applies the linear transformation only to the last dimension (the features dimension),
#  preserving all other dimensions.

model = MyModule(num_inputs=13, num_hidden= 2, num_outputs=3)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
    for batch_t, batch_t1 in dataloader:
        pred = model(batch_t)
        loss = criterion(pred, batch_t1)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print(str(loss.item()))
        print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 5172.947265625
Epoch 1, Loss: 12125.986328125
Epoch 1, Loss: 1983.1199951171875
Epoch 1, Loss: 9157.8681640625
Epoch 1, Loss: 6659.4033203125
Epoch 1, Loss: 3494.038818359375
Epoch 1, Loss: 14313.0029296875
Epoch 1, Loss: 4955.49072265625
Epoch 1, Loss: 13421.166015625
Epoch 1, Loss: 13124.53125
Epoch 1, Loss: 5060.3935546875
Epoch 2, Loss: 5049.29931640625
Epoch 2, Loss: 13624.6962890625
Epoch 2, Loss: 15959.568359375
Epoch 2, Loss: 5016.31103515625
Epoch 2, Loss: 19140.380859375
Epoch 2, Loss: 4555.83984375
Epoch 2, Loss: 6959.1943359375
Epoch 2, Loss: 3811.620849609375
Epoch 2, Loss: 4884.052734375
Epoch 2, Loss: 4948.6806640625
Epoch 2, Loss: 894.0061645507812
Epoch 3, Loss: 6221.40185546875
Epoch 3, Loss: 3898.839599609375
Epoch 3, Loss: 3904.694091796875
Epoch 3, Loss: 11111.44140625
Epoch 3, Loss: 4888.6669921875
Epoch 3, Loss: 4878.6123046875
Epoch 3, Loss: 6276.03271484375
Epoch 3, Loss: 16411.578125
Epoch 3, Loss: 12697.9560546875
Epoch 3, Loss: 2898.8662109375

In [14]:
#INTENTO DE EXPORTAR A ONNX
import sys
print(sys.executable)

import onnx
import onnxruntime

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", onnxruntime.__version__)

# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (batch_t)
onnx_program = torch.onnx.export(model, example_inputs, dynamo=True)

onnx_program.save("onnxModels/trainedModel.onnx")

c:\Users\mikel\miniconda3\envs\dl2024\python.exe
ONNX version: 1.21.0
ONNX Runtime version: 1.24.4
[torch.onnx] Obtain model graph for `MyModule([...]` with `torch.export.export`...
[torch.onnx] Obtain model graph for `MyModule([...]` with `torch.export.export`... ✅
[torch.onnx] Translate the graph into ONNX...


W0403 20:35:40.373000 2140 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0403 20:35:40.373000 2140 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0403 20:35:40.373000 2140 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0403 20:35:40.380000 2140 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'flo

[torch.onnx] Translate the graph into ONNX... ✅


In [11]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
